# 02. Building a Tool-Calling Agent with LangChain's `create_agent`

This notebook builds a small operations assistant for a fictional company, "CloudXeus," that can answer questions about order status and product inventory by calling Python functions exposed as LangChain **tools**. It uses `langchain.agents.create_agent`, LangChain's prebuilt ReAct-style tool-calling agent constructor, on top of the same Azure AI Foundry model endpoint from notebook 01. It belongs to Section 04, "Agent frameworks on top of Azure AI Foundry."

**Difficulty: Intermediate**

As in notebook 01: `create_agent`, `@tool`, and the whole LangChain agent loop are third-party framework concepts, **not** part of the AI-102 exam. The AI-102-relevant substance underneath is still just "a chat-completions call against a Foundry deployment" — tool-calling is a wire-protocol feature of the model API itself (the model returns `tool_calls`; something has to execute them and feed results back), and LangChain is simply automating that loop for you.

## Prerequisites

**pip3 packages:**
```bash
pip3 install langchain langchain-openai python-dotenv
```
(`langchain`, `langchain-core`, `langchain-openai` are already in the repo root `requirements.txt`.)

**Azure resources required:**
- Same as notebook 01: an Azure AI Foundry project with a chat-completion model deployed.

**Environment variables** (same as notebook 01 — add to root `.env`):
```
AZURE_AI_OPENAI_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/openai/v1
AZURE_AI_OPENAI_API_KEY=<your-foundry-api-key>
AZURE_AI_MODEL_DEPLOYMENT=<your-chat-deployment-name>
```
No new Azure resources beyond the model deployment — the "tools" in this notebook are plain local Python functions with in-memory dictionaries, not real Azure services.

## What You'll Learn

- How to define a LangChain tool with the `@tool` decorator and a docstring the model reads as the tool's description
- How `create_agent` wires a model + a list of tools + a system prompt into a runnable agent
- The message-based invocation contract (`{"messages": [...]}`) shared by LangChain agents and LangGraph graphs
- How the model decides which tool(s) to call and how results get folded back into the final answer

### Step 1 — Imports, configuration, and the base model

Same setup as notebook 01: load endpoint/key/deployment from environment variables and construct a `ChatOpenAI` client against the Foundry `/openai/v1` endpoint. This `model` object is the same one from notebook 01 — a tool-calling agent is just this model plus a loop that intercepts and executes its tool calls.

💡 **Framework note:** the underlying Azure OpenAI-compatible endpoint is unaware of "agents" at all — from its point of view, tool-calling is just a request that includes a `tools` schema and a response that may include `tool_calls` instead of (or alongside) text. Everything "agentic" here is client-side orchestration performed by LangChain, not a server-side Azure feature.

🔄 **Alternatives:** you could reach the same endpoint with the raw `openai` SDK and pass a `tools=[...]` JSON schema directly to `chat.completions.create`, then manually inspect `response.choices[0].message.tool_calls` and loop — this is exactly the pattern used in `11_azure_ai_foundry/04_prompt_agent/main.py`, which implements the tool-calling loop by hand instead of via a framework.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

endpoint = os.getenv("AZURE_AI_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT")
api_key = os.getenv("AZURE_AI_OPENAI_API_KEY")

model = ChatOpenAI(
    base_url=endpoint,
    api_key=api_key,
    model=deployment_name,
)


### Step 2 — Define the tools

Each `@tool`-decorated function becomes something the model can choose to call. LangChain turns the function's **type hints** and **docstring** into the JSON schema sent to the model, so the docstring isn't just documentation here — it's the description the model reads to decide when this tool is relevant. Both tools below look up an ID in an in-memory dictionary and return a plain string; a real deployment would replace these bodies with calls to an actual order-management or inventory API.

💡 **Framework note:** on the wire, each `@tool` becomes one entry in the `tools` array of the chat-completions request, in the same JSON-Schema-based function-calling format used by Azure OpenAI directly. There's nothing Azure-specific about the tool definitions themselves.

🔄 **Alternatives:** for tools that call *actual* Azure services (e.g., a real order database via Azure SQL, or a lookup via Azure AI Search), you'd keep the `@tool` wrapper but replace the dictionary lookup with a real client call — the agent loop doesn't change.

In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """Get the current status of a CloudXeus order by order ID."""
    orders = {
        "ORD-001": "Dispatched — arriving tomorrow.",
        "ORD-002": "Processing — not yet shipped.",
        "ORD-003": "Delivered on June 12, 2026.",
    }
    return orders.get(order_id, f"Order {order_id} not found.")


@tool
def get_inventory(product_id: str) -> str:
    """Check the available inventory for a CloudXeus product by product ID."""
    inventory = {
        "PRD-A1": "142 units in stock.",
        "PRD-B2": "0 units — out of stock.",
        "PRD-C3": "37 units in stock.",
    }
    return inventory.get(product_id, f"Product {product_id} not found.")


### Step 3 — Build the agent

`create_agent` wraps the model, the tool list, and a system prompt into a single runnable agent object. Internally it builds a small graph (LangChain's `create_agent` is implemented on top of LangGraph) that repeatedly calls the model, executes any requested tools, feeds results back, and loops until the model stops requesting tools.

💡 **Framework note:** this hides exactly the loop that `11_azure_ai_foundry/04_prompt_agent/main.py` writes out by hand (check `message.tool_calls`, execute the matching Python function, append a `role="tool"` message, call the model again). Both reach the same underlying Azure endpoint; `create_agent` just automates the plumbing.

🔄 **Alternatives:** for full control over the loop (e.g., custom retry logic, human-in-the-loop approval before executing a tool, or capping the number of tool round-trips), write the loop manually instead of using `create_agent` — see notebook 04 (`cloudxeus_support_agent.py`) for a case that uses LangGraph's lower-level `StateGraph` for exactly this kind of control.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_order_status, get_inventory],
    system_prompt=(
        "You are a helpful CloudXeus operations assistant. "
        "Use the available tools to answer questions accurately."
    ),
)


### Step 4 — Invoke the agent

Agents built with `create_agent` are invoked with a `messages` list, mirroring the OpenAI chat message format. The final response is also a `messages` list — the last entry is the agent's answer after it has (possibly) called tools and incorporated their results.

💡 **Framework note:** `response["messages"]` contains the *full* transcript, including any intermediate tool-call and tool-result messages the agent generated along the way — inspect it if you want to see exactly which tools were invoked and with what arguments.

🔄 **Alternatives:** call `agent.stream(...)` instead of `.invoke(...)` to observe each step (model call, tool call, tool result, final answer) as it happens, which is useful for debugging why an agent chose a particular tool.

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the status of order ORD-002 and how many units of PRD-A1 do we have?",
            }
        ]
    }
)

print(response["messages"][-1].content)


## Summary

This notebook defined two local tools (`get_order_status`, `get_inventory`), wired them into a `create_agent`-built LangChain agent alongside a system prompt, and asked a question that requires calling *both* tools. The agent's model decided which tools to call, LangChain executed them, and the final printed message is the model's answer synthesized from both tool results — a two-hop tool-calling round trip in a single `.invoke()` call.

## Try It Yourself

1. **Easy:** Add a third tool, e.g. `get_shipping_estimate(destination: str) -> str`, and ask a question that requires all three tools in one query.
2. **Intermediate:** Inspect `response["messages"]` in full (not just the last message) and print each message's `role` and content/tool-call info to see the exact sequence of model → tool → model steps the agent took.
3. **Advanced:** Replace `create_agent` with a hand-written loop using `model.bind_tools([...])` and manual `tool_calls` inspection (mirroring `11_azure_ai_foundry/04_prompt_agent/main.py`'s pattern), and compare the amount of code required against the one-line `create_agent` call.